# C3.2 Practice — Make Your HDB Model Better & More Trustworthy

**Module 3 · Machine Learning & GenAI — Coaching Add-on**

---

### The story so far
In C3.1 you built a model that predicts an HDB flat's resale price from **3 things**: floor area, lease year, and floor level. You deployed it in Streamlit. 

Today's question: **Is it any good? And can we make it better?**

### Your mission (about 1 hour)
1. **Measure** how wrong your current model is, in plain dollars.
2. **Add features** and see the error shrink.
3. **Compare two models** and learn how to pick one honestly.

> 💡 **How to use this notebook:** Run each cell with `Shift + Enter`. Cells marked **🔧 YOUR TURN** have blanks (`____`) for you to fill in. Stuck? Open the **▸ Hint** below it. Full answers are at the very bottom.

**Scenario:** You're a junior analyst at a property agency. Your manager says: *"Nice predictor — but how much can I trust the number it gives a client?"* Let's find out.

In [1]:
# Setup — run this first (needs internet to download the data)
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

url = ("https://raw.githubusercontent.com/flexfengfeng/6m-data-C3.2/"
       "main/notebooks/data/"
       "Resale_flat_prices_based_on_registration_date_from_Jan-2017_onwards.csv")
data = pd.read_csv(url)
data["floor_level"] = data["storey_range"].str.split(" ").str[0].astype(float)
print("Rows of real HDB sales loaded:", len(data))
data.head()

Rows of real HDB sales loaded: 233479


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,floor_level
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,10.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,1.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,1.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,4.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,1.0


> 📊 **Reading the output:** the print line confirms how many real HDB resale records were downloaded (about 233,000 rows), and `data.head()` shows the first 5 rows so you can eyeball the columns. Check that `floor_level` appears as a number on the right — that's the column we just built from `storey_range` (e.g. `"10 TO 12"` → `10.0`). If you see the table, the data loaded correctly and you're ready for Part A.

In [2]:
print("Data types:")
print(data.dtypes)
print()
print("Missing values:")
missing = data.isna().sum()
print(missing[missing > 0])
print()
print("Numerical summary (excluding missing):")
print(data.describe(include="number").round(2))

Data types:
month                      str
town                       str
flat_type                  str
block                      str
street_name                str
storey_range               str
floor_area_sqm         float64
flat_model                 str
lease_commence_date      int64
remaining_lease            str
resale_price           float64
floor_level            float64
dtype: object

Missing values:
Series([], dtype: int64)

Numerical summary (excluding missing):
       floor_area_sqm  lease_commence_date  resale_price  floor_level
count       233479.00            233479.00     233479.00    233479.00
mean            96.69              1996.55     531028.39         7.78
std             24.02                14.36     190174.93         5.95
min             31.00              1966.00     140000.00         1.00
25%             81.00              1985.00     390000.00         4.00
50%             93.00              1997.00     500000.00         7.00
75%            112.00         

## 📏 Three scorecards, in plain English (no stats needed)

From here on we judge every model with three numbers. Here's what they really mean:

**MAE — "on average, how many dollars off?"**
Imagine the model guesses the price of 100 flats. For each one, you check how far the guess was from the true price (ignore whether it guessed too high or too low — just the size of the miss). Average those misses, and that's the MAE.
- MAE = S\$80,000 → "typically, the guess is about \$80k away from reality."
- **Lower is better.** It's in plain dollars, so anyone can understand it.

**MAPE — "on average, how many *percent* off?"**
Same idea as MAE, but each miss is measured *relative to that flat's price* instead of in raw dollars. A \$50k miss on a \$1.2M flat is small; the same \$50k miss on a \$300k flat is huge. MAPE captures that by turning every miss into a percentage, then averaging.
- MAPE = 15% → "typically, the guess is about 15% away from the true price."
- **Lower is better.** It's handy because it's fair across cheap and expensive flats, and easy to say out loud ("we're usually within ~15%").

**R² — "out of everything that makes prices differ, how much did the model figure out?"**
Flats sell for wildly different prices. Some of that is explainable (size, location, flat type…) and a good model should capture it. R² is the *fraction* of that price variation the model successfully explains, written as a number from 0 to 1:
- **R² = 0** → useless: no better than ignoring the flat and just guessing the average price every time.
- **R² = 0.70** → "the model accounts for about 70% of why flats differ in price." (The other 30% is stuff it can't see — renovations, the view, how keen the buyer was, luck.)
- **R² = 1.0** → perfect: every prediction dead-on (never happens in the real world).
- **Higher is better.**

**Why use all three?** MAE gives the error *in dollars you can feel*; MAPE gives it *as a percentage* that's fair across cheap and pricey flats; R² gives *what share of the puzzle* the model solved. Together they give the honest picture. As you add features and change models below, watch MAE and MAPE go **down** and R² go **up**.

## Part A — How wrong is the current (3-feature) model?

We'll rebuild your L05 model and, this time, **score** it.

**Key idea — train/test split:** We teach the model on 80% of the flats, then quietly test it on the other 20% it has *never seen*. That's the honest way to check it — like setting exam questions a student hasn't already memorised.

**Key idea — MAE (Mean Absolute Error):** the average gap between the predicted price and the real price. If MAE = \$60,000, the model is *on average* \$60k off. Lower is better.

**Key idea — MAPE (Mean Absolute Percentage Error):** the same average gap, but as a *percentage* of each flat's price. If MAPE = 15%, the model is *on average* 15% off — fair to compare across cheap and pricey flats. Lower is better.

**Key idea — R² (R-squared):** the share of the price variation the model explains, from 0 to 1. R² = 0 means it's no better than always guessing the average price; R² = 1 means perfect. Higher is better — it's a quick "how much of the picture did we capture?" score that complements the dollar/percentage views.

### ✋ Pause & Predict
Before running: with only 3 features, do you think the model will be off by closer to **\$20k**, **\$60k**, or **\$120k**? Write your guess, then run the cell.

In [3]:
# Part A — baseline model with the original 3 features
features_3 = ["floor_area_sqm", "lease_commence_date", "floor_level"]
X = data[features_3]
y = data["resale_price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = LinearRegression()
baseline.fit(X_train, y_train)
preds = baseline.predict(X_test)

mae_baseline = mean_absolute_error(y_test, preds)
mape_baseline = mean_absolute_percentage_error(y_test, preds)
r2_baseline = r2_score(y_test, preds)
print(f"Baseline (3 features): on average off by S${mae_baseline:,.0f}")
print(f"Baseline (3 features): MAPE = {mape_baseline:.1%}  (average % off — fair across cheap & pricey flats)")
print(f"Baseline (3 features): R² = {r2_baseline:.3f}  (share of price variation explained, 1.0 = perfect)")

Baseline (3 features): on average off by S$102,499
Baseline (3 features): MAPE = 20.1%  (average % off — fair across cheap & pricey flats)
Baseline (3 features): R² = 0.500  (share of price variation explained, 1.0 = perfect)


> 📊 **Reading the output:** with just 3 features the model is on average **~S\$102,000 off (MAE)**, which is about **~20% off (MAPE)**, and its **R² is ~0.50** — it explains only about half of why prices differ between flats. That's a weak-but-not-useless starting score: clearly a lot of what drives price (location, flat type, …) is still missing. Write all three numbers down — every later part is measured against them.

**What do you notice?** That dollar figure is your starting score. Every change you make from here, you'll compare against it. Write it down.

## Part B — 🔧 YOUR TURN: Add more features

Right now the model knows nothing about **what kind of flat** it is or **where** it is. A 5-room in Bishan and a 2-room in Yishun of the same size get treated the same — no wonder it struggles.

We'll add two columns: `flat_type` and `town`. These are **labels**, not numbers, so we convert them into 0/1 columns with `pd.get_dummies()` (a flat either *is* "4 ROOM" or it isn't).

In [4]:
# Part B — add flat_type and town
numeric = ["floor_area_sqm", "lease_commence_date", "floor_level"]
category = ["flat_type", "town"]

# 🔧 YOUR TURN: build X_more by combining the numeric + category columns,
#    then turning the categories into 0/1 columns.
X_more = pd.get_dummies(data[numeric + category], columns=category)   # <- fill the blank
y = data["resale_price"]

X_train, X_test, y_train, y_test = train_test_split(X_more, y, test_size=0.2, random_state=42)

richer = LinearRegression()
richer.fit(X_train, y_train)
richer_preds = richer.predict(X_test)
mae_richer = mean_absolute_error(y_test, richer_preds)
mape_richer = mean_absolute_percentage_error(y_test, richer_preds)
r2_richer = r2_score(y_test, richer_preds)

print(f"Baseline (3 features): MAE S${mae_baseline:,.0f}   MAPE {mape_baseline:.1%}   R² {r2_baseline:.3f}")
print(f"Richer  (5 features): MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
print(f"Improvement: S${mae_baseline - mae_richer:,.0f} smaller error, MAPE {mape_baseline - mape_richer:+.1%}, R² up {r2_richer - r2_baseline:+.3f}")

Baseline (3 features): MAE S$102,499   MAPE 20.1%   R² 0.500
Richer  (5 features): MAE S$82,761   MAPE 16.7%   R² 0.704
Improvement: S$19,738 smaller error, MAPE +3.4%, R² up +0.205


> 📊 **Reading the output:** adding `flat_type` and `town` drops the error from **~S\$102k to ~S\$83k (MAE)**, tightens **MAPE from ~20% to ~17%**, and lifts **R² from ~0.50 to ~0.70**. So telling the model *what kind* of flat it is and *where* it sits explains an extra ~20% of price variation — a big, cheap win. Same model (Linear Regression), just more relevant information. This is the single most important lesson of the lab: better features often beat fancier models.

<details><summary>▸ Hint</summary>

`get_dummies` needs to know which columns are categories. You already stored them in the variable `category`. So the blank is just `category`.
</details>

**What do you notice?** Did the error drop? By how much? More relevant information usually = better predictions — but not always, which is what Part C explores.

## Part C — 🔧 YOUR TURN: Choose a model + check for overfitting

Linear Regression draws straight-line relationships. A **Random Forest** can capture curves and combinations (e.g. "top floor *in a central town* is worth extra"). Let's see if it does better.

**Watch out for overfitting:** a model can score brilliantly on flats it trained on but flop on new ones — like a student who memorised past papers but can't handle new questions. We catch this by comparing the **training score** with the **test score**. A big gap = memorising, not learning.

In [5]:
# Part C — compare Linear Regression vs Random Forest
# 🔧 YOUR TURN: create a Random Forest model in the blank.
forest = RandomForestRegressor(n_estimators=100, random_state=42)   # <- fill the blank
forest.fit(X_train, y_train)

# Test-set error (on unseen flats) vs training-set error (on flats it learned from)
mae_test   = mean_absolute_error(y_test,  forest.predict(X_test))
mae_train  = mean_absolute_error(y_train, forest.predict(X_train))
mape_test  = mean_absolute_percentage_error(y_test,  forest.predict(X_test))
mape_train = mean_absolute_percentage_error(y_train, forest.predict(X_train))
r2_test    = r2_score(y_test,  forest.predict(X_test))
r2_train   = r2_score(y_train, forest.predict(X_train))

print(f"Linear Regression  test: MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
print(f"Random Forest      test: MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print()
print(f"Random Forest  TRAIN: MAE S${mae_train:,.0f}   MAPE {mape_train:.1%}   R² {r2_train:.3f}")
print(f"Random Forest   TEST: MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print(f"Gap (overfitting signal): MAE S${mae_test - mae_train:,.0f}, MAPE {mape_test - mape_train:+.1%}, R² {r2_train - r2_test:+.3f}")

Linear Regression  test: MAE S$82,761   MAPE 16.7%   R² 0.704
Random Forest      test: MAE S$72,629   MAPE 14.4%   R² 0.774

Random Forest  TRAIN: MAE S$62,456   MAPE 12.6%   R² 0.833
Random Forest   TEST: MAE S$72,629   MAPE 14.4%   R² 0.774
Gap (overfitting signal): MAE S$10,173, MAPE +1.8%, R² +0.059


> 📊 **Reading the output:** the Random Forest beats Linear Regression on the unseen test set — error falls to **~S\$73k (MAE)**, **~14% (MAPE)**, and **R² rises to ~0.77** — because it can capture curves and feature combinations a straight line can't. But look at the train-vs-test gap: it scores **better on flats it trained on** (MAE ~S\$62k, MAPE ~13%, R² ~0.83) than on new ones (MAE ~S\$73k, MAPE ~14%, R² ~0.77). That gap *is* overfitting — the model has partly memorised the training flats. The gap here is modest, so the forest is still the better model to ship; always judge by the **test** score, never the train score.

<details><summary>▸ Hint</summary>

We imported it at the top as `RandomForestRegressor`. Put that in the blank.
</details>

**What do you notice?**
- Which model has the lower **test** error? That's usually the one to ship.
- Is the Random Forest's **train** error much smaller than its **test** error? That gap is overfitting — it knows the training flats almost perfectly but is less sharp on new ones.

## 🏆 Challenge (if you have time)
Pick **one**:
1. Add a third feature of your choice (e.g. `remaining_lease` cleaned to a number) and see if the error drops further.
2. Try `RandomForestRegressor(n_estimators=300)` — does more trees help the test error, or just slow it down?
3. Find the single flat in the test set where the model was **most wrong**. What was special about it?

## Part D — 🔧 (Stretch) Stacking: can a *team* of models beat the best one?

Your learner asked a great question: can we combine Random Forest and Gradient Boosting?

The feasible way to do this is **stacking** — train several different models *side by side*, then let one small final model (the "manager") learn how to best **blend** their predictions. The idea: each model makes different mistakes, so a smart blend can beat any single one.

> ⚠️ **Common misconception:** you do **not** chain them so Random Forest "narrows down" and Gradient Boosting "fine-tunes" the same price. Each model already predicts the full price on its own — you blend them, you don't hand one's answer to the other. (The one model that *does* "fine-tune its own mistakes step by step" is Gradient Boosting itself, internally.)

**Analogy:**
- Random Forest = average 100 independent guesses.
- Gradient Boosting = each guesser fixes the previous one's mistakes, in sequence.
- **Stacking** = collect guesses from several experts, then a manager learns whose opinion to trust when.

Let's test whether the extra complexity is actually worth it for our HDB data.

### ✋ Pause & Predict
The stacked team uses *more* computation than a single model. Do you think it will **beat** the best single model's test error, or only **tie** it for a lot more effort? Write your guess, then run.

In [6]:
# Part D — stack Random Forest + Gradient Boosting, blended by a Linear Regression "manager"
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor

# Two different base models (they make different kinds of mistakes)
base_models = [
    ("random_forest", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("grad_boost",    GradientBoostingRegressor(random_state=42)),
]

# 🔧 YOUR TURN: the "manager" that learns how to blend the base models.
#    Use a simple LinearRegression() as the final_estimator.
stack = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression(),          # <- fill the blank
)
stack.fit(X_train, y_train)
stack_preds = stack.predict(X_test)
mae_stack  = mean_absolute_error(y_test, stack_preds)
mape_stack = mean_absolute_percentage_error(y_test, stack_preds)
r2_stack   = r2_score(y_test, stack_preds)

# Also score Gradient Boosting on its own, for a fair comparison
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)
mae_gb  = mean_absolute_error(y_test, gb_preds)
mape_gb = mean_absolute_percentage_error(y_test, gb_preds)
r2_gb   = r2_score(y_test, gb_preds)

print(f"Linear Regression (Part B): MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
print(f"Random Forest    (Part C): MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print(f"Gradient Boosting (alone) : MAE S${mae_gb:,.0f}   MAPE {mape_gb:.1%}   R² {r2_gb:.3f}")
print(f"Stacked team             : MAE S${mae_stack:,.0f}   MAPE {mape_stack:.1%}   R² {r2_stack:.3f}")

Linear Regression (Part B): MAE S$82,761   MAPE 16.7%   R² 0.704
Random Forest    (Part C): MAE S$72,629   MAPE 14.4%   R² 0.774
Gradient Boosting (alone) : MAE S$78,854   MAPE 15.7%   R² 0.725
Stacked team             : MAE S$71,903   MAPE 14.3%   R² 0.781


> 📊 **Reading the output:** Gradient Boosting alone lands around **~S\$79k (MAE), ~16% (MAPE), R² ~0.73**, and the **stacked team** comes in around **~S\$72k, ~14%, R² ~0.78** — essentially tied with the single Random Forest from Part C on all three scorecards. So blending three models bought us almost nothing here, despite ~3× the training work. That's the takeaway: stacking is a real competition-winning trick, but on data like this a single good model is usually within a whisker, so reach for the extra complexity only when that last sliver of accuracy genuinely pays off.

<details><summary>▸ Hint</summary>

The "manager" is just another model. We've used it all along: `LinearRegression()`. Put that in the blank.
</details>

**What do you notice?**
- Did the **stacked team** beat the best single model? By how many dollars?
- Was the improvement big enough to justify training *three* models instead of one?
- Often a single well-tuned Gradient Boosting model is within a whisker of the stack — which is why, in practice, you usually reach for stacking only when that last bit of accuracy really matters (it's a classic competition-winning trick, but frequently overkill).

## Part E — 💡 So how *do* we push the error down further?

A learner asked the natural follow-up: *"We've tried more features, a Random Forest, and stacking — what else can we do?"*

There are really only **three levers** for a better model, and they're worth knowing in order of bang-for-buck:

1. **🥇 Better features (feature engineering)** — give the model more *relevant* information. This was our biggest win in Part B (S\$102k → S\$83k just by adding `flat_type` and `town`). We left an obvious one on the table: `remaining_lease` (how many years are left on the 99-year lease). It's stored as text like `"61 years 04 months"`, so we have to clean it into a number first.
2. **🥈 Tuning the model (hyperparameters)** — keep the same model but adjust its dials. For Gradient Boosting the big ones are `n_estimators` (how many correction steps) and `learning_rate` (how big each step is). More, smaller steps usually generalise better — at the cost of training time.
3. **🥉 More data / a stronger model** — usually the last and most expensive resort.

The cell below pulls levers **1 and 2 together**: it engineers a `remaining_lease_years` feature *and* uses a more carefully tuned Gradient Boosting model, then checks whether the test error actually drops.

In [7]:
# Part E — push further: (1) engineer a new feature, (2) tune the model

# --- Lever 1: feature engineering --------------------------------------------
# 'remaining_lease' is text like "61 years 04 months". Turn it into a single
# number of years (months count as a fraction) so the model can use it.
def lease_to_years(text):
    parts = text.split()
    years = int(parts[0])
    months = int(parts[2]) if "month" in text else 0
    return years + months / 12

data["remaining_lease_years"] = data["remaining_lease"].apply(lease_to_years)

# Same features as Part B, plus our new engineered one.
numeric_plus = ["floor_area_sqm", "lease_commence_date", "floor_level", "remaining_lease_years"]
category = ["flat_type", "town"]
X_plus = pd.get_dummies(data[numeric_plus + category], columns=category)
y = data["resale_price"]

# IMPORTANT: same random_state=42 split as before, so the comparison is fair.
X_train, X_test, y_train, y_test = train_test_split(X_plus, y, test_size=0.2, random_state=42)

# --- Lever 2: a more carefully tuned Gradient Boosting model -----------------
# More trees + a smaller learning_rate = many small, careful correction steps.
tuned_gb = GradientBoostingRegressor(
    n_estimators=400,      # more correction steps (default is 100)
    learning_rate=0.05,    # smaller steps generalise better
    max_depth=4,           # slightly deeper trees catch feature combinations
    random_state=42,
)
tuned_gb.fit(X_train, y_train)
tuned_preds = tuned_gb.predict(X_test)

mae_tuned  = mean_absolute_error(y_test, tuned_preds)
mape_tuned = mean_absolute_percentage_error(y_test, tuned_preds)
r2_tuned   = r2_score(y_test, tuned_preds)

print("Best so far (Random Forest, Part C):")
print(f"    MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print("Tuned Gradient Boosting + extra feature (Part E):")
print(f"    MAE S${mae_tuned:,.0f}   MAPE {mape_tuned:.1%}   R² {r2_tuned:.3f}")
print()
print(f"Change vs best so far: MAE S${mae_test - mae_tuned:,.0f} smaller, "
      f"MAPE {mape_tuned - mape_test:+.1%}, R² {r2_tuned - r2_test:+.3f}")

Best so far (Random Forest, Part C):
    MAE S$72,629   MAPE 14.4%   R² 0.774
Tuned Gradient Boosting + extra feature (Part E):
    MAE S$42,803   MAPE 8.3%   R² 0.908

Change vs best so far: MAE S$29,826 smaller, MAPE -6.0%, R² +0.134


> 📊 **Reading the output:** this is the biggest leap in the whole notebook — error drops from **~S\$73k to ~S\$43k (MAE)**, **MAPE from ~14% to ~8%**, and **R² jumps from ~0.77 to ~0.91**. The model now explains ~90% of why flats differ in price. Most of that win comes from **lever 1**: `remaining_lease_years` is one of the strongest price signals in HDB data (buyers pay a lot of attention to lease decay), and we'd been throwing it away as messy text. The tuning (lever 2) adds a smaller polish on top.
>
> **The lesson, ranked:** when a model isn't good enough, reach for a *better feature* before a *fancier model* almost every time — it's cheaper and usually wins by more. Tuning is the fine-tuning pass you do once the features are right. (And note the honesty caveat: `remaining_lease` is basically a cleaner version of `lease_commence_date`, so always sanity-check that a powerful new feature isn't secretly leaking the answer — here it's legitimate, since it's known at sale time.)

## Part F — 💾 How big is each model when you save it as a `.pkl`?

A learner asked: *"If I save every model in this notebook to a `.pkl` file, how big is each one — and why are they so different?"*

Great question, because **file size = how much the model has to memorise to make a prediction.** A model is saved (`pickle`d) so you can load it later in `app.py` without retraining. The cell below pickles each model we trained above and reports its size. The numbers below are the real measured sizes on this dataset.

In [ ]:
# Part F — measure the saved (.pkl) size of every model we built
import pickle

def pkl_size(model):
    """Bytes this model takes when pickled (same as a .pkl file on disk)."""
    return len(pickle.dumps(model))

def human(n):
    for unit in ("B", "KB", "MB"):
        if n < 1024 or unit == "MB":
            return f"{n:,.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024

models = [
    ("A. Linear Regression (3 features)", baseline),
    ("B. Linear Regression (5 feat + dummies)", richer),
    ("C. Random Forest (100 trees)", forest),
    ("D. Gradient Boosting (100 steps, default)", gb),
    ("D. Stacking (RF + GB + LR manager)", stack),
    ("E. Tuned Gradient Boosting (400 steps, depth 4)", tuned_gb),
]

rows = []
for name, m in models:
    b = pkl_size(m)
    rows.append({"Model": name, "Bytes": b, "Size": human(b)})

sizes = pd.DataFrame(rows).sort_values("Bytes").reset_index(drop=True)
sizes

> 📊 **Reading the output — the sizes span a factor of ~500,000×:**
>
> | Model | `.pkl` size | What's stored inside |
> |---|---:|---|
> | A. Linear Regression (3 features) | **~0.6 KB** | 3 slopes + 1 intercept |
> | B. Linear Regression (5 feat + dummies) | **~1.7 KB** | ~36 slopes + 1 intercept |
> | D. Gradient Boosting (100 steps, default) | **~136 KB** | 100 *tiny* trees (depth 3) |
> | E. Tuned Gradient Boosting (400 steps, depth 4) | **~970 KB** | 400 small trees (depth 4) |
> | C. Random Forest (100 trees) | **~328 MB** | 100 *deep, full-grown* trees |
> | D. Stacking (RF + GB + LR) | **~328 MB** | the whole Random Forest + the whole Gradient Boosting + a tiny manager |
>
> ### Why the huge difference?
>
> **A model's file size = how many numbers it has to memorise to predict.** That depends almost entirely on the *type* of model, not its accuracy.
>
> **1. Linear models are tiny (bytes–kilobytes).** A linear regression is just a formula: `price = w₁·area + w₂·lease + … + intercept`. It stores *one number per feature*. 3 features → ~0.6 KB; 36 dummy columns → ~1.7 KB. That's the entire model. This is why the most accurate model isn't the biggest — size tracks *structure*, not skill.
>
> **2. Random Forest is gigantic (~328 MB).** It stores **100 separate decision trees**, and by default each tree is grown **all the way down** until its leaves are nearly pure. On 233,000 training rows that means each tree has *tens of thousands of branch nodes*, and every node stores which feature it splits on, the threshold, and pointers to its children. 100 huge trees × tens of thousands of nodes each = hundreds of megabytes. **The forest essentially memorises the training data inside its branches** — which is also why it overfits a little (Part C) and why it's the largest file here.
>
> **3. Gradient Boosting is small (KB–MB) even with *more* trees.** Counter-intuitively, the tuned GB has **400 trees** but is still ~340× smaller than the 100-tree forest. The reason: boosting trees are deliberately **shallow** (depth 3–4 → at most ~8–16 leaves each). Each one only makes a small correction, so it stays tiny. 400 shallow trees still add up to far less than 100 full-depth ones. Depth matters far more than count: the default GB (100 × depth-3) is ~136 KB; bumping to 400 × depth-4 grows it to ~970 KB — bigger, but still under 1 MB.
>
> **4. Stacking is the sum of its parts (~328 MB).** A `StackingRegressor` literally stores *every* base model inside it — here the full Random Forest **plus** the full Gradient Boosting **plus** the little Linear Regression manager. So its size ≈ forest (328 MB) + GB (0.14 MB) + a few bytes. Combining models multiplies storage as well as compute.
>
> ### The practical takeaway
> - **Size is driven by model *structure* (how many trees × how deep), not by accuracy.** Part E's tuned GB is both the **most accurate** *and* ~340× **smaller** than the Random Forest — a great thing to ship in `app.py`.
> - A 328 MB Random Forest is slow to load and awkward to deploy. If two models tie on accuracy, prefer the smaller one.
> - To shrink a Random Forest, cap `max_depth` or `min_samples_leaf` (stop the trees growing so deep) — you'll trade a sliver of accuracy for a massive size reduction.

## ✅ Wrap-up

| Idea | In one line |
|---|---|
| **MAE** | Average dollar error — your model's report card. |
| **Train/test split** | Test on unseen flats for an honest score. |
| **Adding features** | More relevant info usually shrinks the error. |
| **Model choice** | Pick the lowest **test** error, not training error. |
| **Overfitting** | Big train-vs-test gap = memorising, not learning. |

**Up next:** plug your better model into the upgraded `app.py` and redeploy in Streamlit.

---
## Sample solutions (look only after trying!)

**Part B blank:** `X_more = pd.get_dummies(data[numeric + category], columns=category)`

**Part C blank:** `forest = RandomForestRegressor(n_estimators=100, random_state=42)`


**Part D blank:** `final_estimator=LinearRegression()`